# Notebook 05 — SFT Fundamentals

> **阶段**：Stage 3 Train · **预计时间**：60–120 分钟（CPU 冒烟） · **平台**：Kaggle Notebook

重点不是刷分，而是理解 Supervised Fine-Tuning 到底改变了什么。


# Learning Objectives

- 构造 SFT 训练样本（图像 + 指令 + 目标 DocTags）并理解 label mask；
- 走通 Dataset → Processor → Collator → Forward → Loss → Backward → Optimizer；
- 完成一次小规模教学 SFT 并记录 training/validation loss 与资源消耗；
- 回答：loss 下降是否等于 Document Parsing 变好？


# Why This Matters

SFT 是理解后续 LoRA、GRPO 等一切后训练方法的基础。先在小数据上观察 loss 行为与 before/after 差异，才能判断训练是否有意义。


# Concepts

```text
Dataset
   ↓
Processor（图像 + chat template）
   ↓
Collator（padding / label mask）
   ↓
Forward → Loss（只算 assistant 部分）
   ↓
Backward → Optimizer
```

- **label mask**：prompt 位置置 -100，loss 只监督目标 DocTags；
- **教学数据**：使用 Notebook 02 的 teaching subset（标记 NOT for official claims）；
- **CPU 冒烟**：4 训练样本、max_steps=2；GPU 环境改为 20–100 样本。


## Step 1 — 构造训练样本


In [ ]:

# 仓库路径定位：兼容「Notebook 位于仓库根目录」与「仓库克隆在 /kaggle/working 子目录」
from pathlib import Path
import sys

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "src" / "model.py").exists():
    matches = [p for p in REPO_ROOT.iterdir() if p.is_dir() and (p / "src" / "model.py").exists()]
    if not matches:
        raise FileNotFoundError(
            "未找到仓库根目录。请按 notebooks/README.md 把仓库克隆到 /kaggle/working，"
            "或把本 Notebook 放在仓库根目录。"
        )
    REPO_ROOT = matches[0].resolve()
sys.path.insert(0, str(REPO_ROOT))
print("REPO_ROOT =", REPO_ROOT)

from src import data
from src.prompts import get_prompt

data_root = data.find_dataset_root()
annotations = data.load_annotations(data_root)
split = data.build_teaching_split(annotations, n_train=4, n_val=2, seed=42)
train_records = data.build_sft_records(split['train'], data_root, get_prompt('v0'))
val_records = data.build_sft_records(split['val'], data_root, get_prompt('v0'))
print('train records:', len(train_records), '| val records:', len(val_records))
print('第一条记录目标前 200 字符:')
print(train_records[0]['target_doctags'][:200])


## Step 2 — Dataset / Processor / Collator 与 label mask


In [ ]:

# 仓库路径定位：兼容「Notebook 位于仓库根目录」与「仓库克隆在 /kaggle/working 子目录」
from pathlib import Path
import sys

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "src" / "model.py").exists():
    matches = [p for p in REPO_ROOT.iterdir() if p.is_dir() and (p / "src" / "model.py").exists()]
    if not matches:
        raise FileNotFoundError(
            "未找到仓库根目录。请按 notebooks/README.md 把仓库克隆到 /kaggle/working，"
            "或把本 Notebook 放在仓库根目录。"
        )
    REPO_ROOT = matches[0].resolve()
sys.path.insert(0, str(REPO_ROOT))
print("REPO_ROOT =", REPO_ROOT)

from src.model import SmolDoclingAdapter
from src.training import SFTDataset, collate_fn

adapter = SmolDoclingAdapter().load()
train_ds = SFTDataset(train_records, adapter.processor)
val_ds = SFTDataset(val_records, adapter.processor)

sample = train_ds[0]
print('input_ids shape:', tuple(sample['input_ids'].shape))
print('pixel_values shape:', tuple(sample['pixel_values'].shape))
masked = int((sample['labels'] == -100).sum())
print('label mask 占比: %.1f%%（prompt 部分不计算 loss）' % (100 * masked / sample['labels'].numel()))


## Step 3 — 一次 Forward + Loss（不训练）


In [ ]:

# 仓库路径定位：兼容「Notebook 位于仓库根目录」与「仓库克隆在 /kaggle/working 子目录」
from pathlib import Path
import sys

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "src" / "model.py").exists():
    matches = [p for p in REPO_ROOT.iterdir() if p.is_dir() and (p / "src" / "model.py").exists()]
    if not matches:
        raise FileNotFoundError(
            "未找到仓库根目录。请按 notebooks/README.md 把仓库克隆到 /kaggle/working，"
            "或把本 Notebook 放在仓库根目录。"
        )
    REPO_ROOT = matches[0].resolve()
sys.path.insert(0, str(REPO_ROOT))
print("REPO_ROOT =", REPO_ROOT)

import torch
from src.training import collate_fn

adapter.model.train()
batch = {k: v.to(adapter.device) for k, v in collate_fn([train_ds[0]]).items()}
with torch.set_grad_enabled(True):
    outputs = adapter.model(**batch)
loss = outputs.loss if hasattr(outputs, 'loss') else outputs[0]
print('single-batch loss:', float(loss))


## Step 4 — 教学 SFT（CPU 冒烟：2 steps）


In [ ]:

# 仓库路径定位：兼容「Notebook 位于仓库根目录」与「仓库克隆在 /kaggle/working 子目录」
from pathlib import Path
import sys

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "src" / "model.py").exists():
    matches = [p for p in REPO_ROOT.iterdir() if p.is_dir() and (p / "src" / "model.py").exists()]
    if not matches:
        raise FileNotFoundError(
            "未找到仓库根目录。请按 notebooks/README.md 把仓库克隆到 /kaggle/working，"
            "或把本 Notebook 放在仓库根目录。"
        )
    REPO_ROOT = matches[0].resolve()
sys.path.insert(0, str(REPO_ROOT))
print("REPO_ROOT =", REPO_ROOT)

from src.training import train_sft

result = train_sft(
    adapter.model,
    train_ds,
    val_dataset=val_ds,
    epochs=1,
    lr=1e-4,
    batch_size=1,
    max_steps=2,
    device=adapter.device,
    output_dir=REPO_ROOT / 'results' / 'sft' / 'teaching_smoke',
)
print(result)


# What You Should Observe

- training_summary.json 记录了 loss、steps、lr、wall_sec 与设备（GPU 时含显存）；
- 2 个 step 的 loss 变化没有统计意义——它是链路验证，不是性能证据；
- checkpoint 保存在 results/sft/，Notebook 重启后可从断点继续。


# Research Checkpoint

> **Loss 下降是否意味着 Document Parsing 一定变好？** 给出至少两个反例场景，并说明为什么最终必须回到 Benchmark 与错误分析。

**TODO：** 答案写入 `results/nb05/research_checkpoint.md`。


# Exercises

1. **TODO：** 在 GPU 环境把 n_train 提到 20–100、max_steps 提高，观察 train/val loss 曲线；
2. **TODO：** 训练前后各用 `max_new_tokens=512` 对同一页推理一次，对比 doctags 差异（注意 CPU 上控制在 1 页）；
3. **TODO：** 解释 label mask 的作用：如果不做 mask，loss 会包含什么、可能带来什么行为。


# Takeaways

- SFT 的最小闭环 = 样本 + 掩码 + 训练循环 + 记录；
- loss 曲线是训练健康度信号，不是任务性能证据；
- 教学数据来自 derived subset，与官方 benchmark 隔离。

**下一步**：[Notebook 06](06_LoRA_Fine_Tuning.ipynb) — 参数高效的微调。
